# Kronos-TH Backtest Re-run (GPU)

Parameterized by YEAR. Edit the YEAR cell below to run 2023, 2024, 2025, or 2026.
GPU + Internet ON.

Each run does full precompute (n_samples=50) + walkforward, outputting to `data/backtest_results/thai_equity_{YEAR}_n50_v2/`.
Expect ~8-12 hours per year on a T4.

In [ ]:
# ── PARAMETER: Change this before running ──────────────────────────────
YEAR = "2023"
# Options: "2023", "2024", "2025", "2026"
# ────────────────────────────────────────────────────────────────────────
print(f"Target year: {YEAR}")

In [ ]:
import os, sys, subprocess, json, time
from pathlib import Path

# ── Clone repo ─────────────────────────────────────────────────────────
REPO_URL = "https://github.com/Yutwizard/Kronos_Thai_Retail.git"
REPO_DIR = "/kaggle/working/Kronos_Thai_Retail"

if not Path(REPO_DIR).exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], timeout=120)

os.chdir(REPO_DIR)

# ── Clone Kronos model repo (required by _kronos_bridge.py) ──
KRONOS_DIR = os.path.join(REPO_DIR, "kronos_repo")
if not Path(KRONOS_DIR).exists():
    subprocess.check_call(["git", "clone", "--depth", "1",
                           "https://github.com/shiyu-coder/Kronos.git",
                           KRONOS_DIR], timeout=120)


# ── Install deps ───────────────────────────────────────────────────────
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "pandas", "yfinance", "torch", "transformers"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR])
print("Dependencies installed.")

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import os; os.environ['HF_HUB_OFFLINE'] = '1'
import time
from pathlib import Path
from kth.data.universe import UNIVERSE
from kth.models.kronos_wrapper import KronosTH
from kth.backtest.walkforward import BacktestConfig, precompute_forecasts, run_walkforward

# ── Date range per year ─────────────────────────────────────────────────
end_date = "2026-05-30" if YEAR == "2026" else f"{YEAR}-12-31"

# ── Load model ──────────────────────────────────────────────────────────
tickers = [x for x, _, _ in UNIVERSE['thai_equity']]
k = KronosTH.from_pretrained('NeoQuasar/Kronos-small', device='cuda')

c = BacktestConfig(start_date=f'{YEAR}-01-01', end_date=end_date,
                   lookback=400, pred_len=20, n_samples=50,
                   position_sizing='equal', max_positions=5,
                   min_ticker_history=20)

print(f'{YEAR}: {len(tickers)} tickers, n_samples=50, offline', flush=True)
t0 = time.time()
precompute_forecasts(k, tickers, start_date=c.start_date, end_date=c.end_date,
                     pred_len=c.pred_len, n_samples=c.n_samples, lookback=c.lookback)
print(f'PRECOMPUTE: {(time.time()-t0)/3600:.1f} hrs', flush=True)

r = run_walkforward(c, k, tickers)
o = Path(f'data/backtest_results/thai_equity_{YEAR}_n50_v2')
o.mkdir(parents=True, exist_ok=True)
r.save(str(o))
m = r.metrics
print(f'DONE: Ret={(r.equity_curve.iloc[-1]/r.equity_curve.iloc[0]-1):+.2%} Sharpe={m["sharpe"]:.2f} MaxDD={m["max_drawdown"]:.2f} p={m["p_value"]:.3f}')

In [ ]:
# ── Verify output ──────────────────────────────────────────────────────
output_dir = f"data/backtest_results/thai_equity_{YEAR}_n50_v2"
output_path = Path(output_dir)

if output_path.exists():
    files = list(output_path.iterdir())
    print(f"Output dir: {output_dir}")
    print(f"Files: {len(files)}")
    for f in files[:10]:
        size_mb = f.stat().st_size / 1e6
        print(f"  {f.name} ({size_mb:.1f} MB)")
    if len(files) > 10:
        print(f"  ... and {len(files) - 10} more")
else:
    print(f"WARNING: Output dir {output_dir} not found!")

print(f"\nDone with {YEAR}. Change YEAR in cell 2 and re-run for next year.")

## Run Plan

1. Set `YEAR = "2023"` → Run All (~8-12 hrs)
2. Set `YEAR = "2024"` → Run All (~8-12 hrs)
3. Set `YEAR = "2025"` → Run All (~8-12 hrs)
4. Set `YEAR = "2026"` → Run All (~8-12 hrs)

**Important:** Each run writes to a separate `_n50_v2` directory, so they won't overwrite each other.
The stale `_n50` and `_n50_full` results are preserved for comparison.

**Troubleshooting:**
- GPU not available → Check Runtime → Change runtime type → GPU T4, Internet ON
- Out of memory → Lower `n_samples` in the run script (not recommended — affects results)
- Kaggle session timeout → Kaggle limits to ~9 hrs. If a run exceeds that, split with later start_date
- Results not saving → Verify `data/backtest_results/` is in the output dir
